In [ ]:
import datetime
import os
import time
import pandas as pd
import numpy as np
import yfinance as yf
import random
from glob import glob
from option_chain_downloader import OptionChainDownloader, get_rotating_logger
from option_finder import OptionFinder
from option_data_plotter import *
from indicators import select_date_range

In [ ]:
try:
    logger
except NameError:
    logger = get_rotating_logger("jupyter", f'logs/daily_screener.log')

def compute_emas(df_price, spans=[21, 50]):
    l_o_ema = []
    for span in spans:
        _df_ema = df_price.ewm(span=span, adjust=False).mean()
        _df_ema.columns = pd.MultiIndex.from_tuples([(c, f'ema_{span}') for c in _df_ema.columns])
        l_o_ema.append(_df_ema)
    return pd.concat(l_o_ema, axis=1)

def compute_share_turnover(df_volumes, df_shares_outstanding):
    s_so = df_shares_outstanding.loc[df_volumes.columns].T.iloc[0]
    return df_volumes.div(s_so, axis='columns')

def rank_shareturnover(df_shareturnover, df_quotes):
    df_st = df_shareturnover.tail(1).T.sort_values(by=df_shareturnover.index[-1], ascending=False)
    df_old_standing = df_st.reset_index().drop(columns=[c for c  in df_st.columns if c != 'index'])
    df_old_standing = df_old_standing.rename(columns={'index': 'symbol'}).reset_index().set_index('symbol').rename(columns={'index': 'old_standing'})
    df_vlast = df_quotes.set_index('symbol').loc[:, ['volume']].rename(columns={'volume': 'last'}).T
    df_st0 = compute_share_turnover(df_vlast, df_shares_outstanding).T.sort_values(by='last', ascending=False)
    df_new_standing = df_st0.reset_index().drop(columns=['last']).reset_index().set_index('symbol')
    df_new_standing = df_new_standing.rename(columns={'index': 'new_standing'})
    return df_new_standing.join(df_old_standing)

def days_from_earning_reports(df_quotes):
    '''df_quotes from OptionFinder.get_quote_df()
    returns df with earningQtrReportDate and earningDays'''
    _df = df_quotes.loc[:, ['symbol', 'earningQtrReportDate']]
    today = pd.Timestamp.today().normalize()
    _df['earningDays'] = (pd.to_datetime(_df.earningQtrReportDate) - today).dt.days
    _df = _df[(_df.earningDays <= 60) & (_df.earningDays >= 0)].sort_values(by='earningDays')
    _df['earningDays'] = _df.earningDays.astype(int)
    return _df.set_index('symbol')

In [ ]:
chain_dir = None
quotes_dir = 'quotes'
cookie_file = 'cookie.txt'
ocd = OptionChainDownloader(chain_dir, quotes_dir, cookie_file, logger, strikes=None)
finder = OptionFinder(logger, chain_dir=None, report_dir=None)

#### This needs only one update per day.  Make sure the latest Bollinger data are from yesterday

In [ ]:
latest_bollinger_file = max(glob('output/bollinger*.csv'))
df_boll = pd.read_csv(latest_bollinger_file)
print('Latest bollinger file:', latest_bollinger_file, df_boll.shape)

#### Top symbols with highest 20-day volume MA
Notes:
- Volume Avg values are in dollars not shares,
- Volume Std have been normalized to ratios with Volume Avg

In [ ]:
_dfb = df_boll.sort_values(by='Volume_Avg', ascending=False).set_index('symbol').head(df_boll.shape[0]//2)
symlist = list(_dfb.index)
print(len(symlist))

In [ ]:
df_close = pd.read_csv('output/yf_close.csv', parse_dates=['Date']).set_index('Date').loc[:, symlist].tail(100)
df_volumes = pd.read_csv('output/yf_volumes.csv', parse_dates=['Date']).set_index('Date').loc[:, symlist].tail(100)
print(df_close.index[-1].strftime('%F'), df_volumes.index[-1].strftime('%F'))

In [ ]:
shares_outstanding_csv_file = os.path.expanduser('~/lab/output/shares_outstanding.csv')
df_shares_outstanding = pd.read_csv(shares_outstanding_csv_file).set_index('symbol')

In [ ]:
df_shareturnover = compute_share_turnover(df_volumes.tail(15), df_shares_outstanding)
n_tops = 20
top_turnovers = df_shareturnover.tail(1).T.sort_values(by=df_shareturnover.index[-1], ascending=False).head(n_tops)
px.line(df_shareturnover[top_turnovers.index], height=600, title=f'Top {n_tops} Share Turnover Rates')

### Pull latest quotes from fidelity

In [ ]:
_t0 = time.time()
count = 0
batch_size = 10
for idx in range(0, len(symlist), batch_size):
    count += ocd.parallel_get_data(symlist[idx:idx+batch_size], rps=5)
    print('.', end='')
print(count, 'file downloads requested in', int(time.time() - _t0), 'seconds')

In [ ]:
df_quotes, df_shortint, df_vola = finder.get_quote_df(symlist)
df_quotes['symbol'] = df_quotes['symbol'].str.replace('/', '-', regex=False)
print('quote time:', df_quotes.lastTime.min(), df_quotes.lastTime.max(), 'after', int(time.time() - _t0), 'seconds')
print(list(df_quotes[df_quotes.symbol.str.contains('/')].symbol))

### Save shares outstanding data to csv

In [ ]:
df_quotes.loc[:, ['symbol', 'sharesOutstanding']].to_csv(shares_outstanding_csv_file, index=None)

In [ ]:
rank_shareturnover(df_shareturnover, df_quotes).head(20)

In [ ]:
df_ed = days_from_earning_reports(df_quotes)

In [ ]:
px.bar(df_ed.head(30), y='earningDays')

In [ ]:
px.bar(df_ed.head(60).tail(30), y='earningDays')

In [ ]:
df_last_price = df_quotes.loc[:, ['symbol', 'lastPrice']].set_index('symbol').rename(columns={'lastPrice': pd.Timestamp.now().normalize()})
df_price = pd.concat([df_close, df_last_price.T])

In [ ]:
df_ema = compute_emas(df_price)
declining = [_symbol for _symbol in symlist if (lambda x: np.all(x.ema_21 < x.ema_50))(df_ema[_symbol].tail(5))]
ascending = [_symbol for _symbol in symlist if (lambda x: np.all(x.ema_21 > x.ema_50))(df_ema[_symbol].tail(5))]
mixed = [_symbol for _symbol in symlist if (lambda x: ~np.all(x.ema_21 < x.ema_50) & ~np.all(x.ema_21 > x.ema_50))(df_ema[_symbol].tail(5))]
print(len(declining), 'declining,', len(ascending), 'ascending,', len(symlist) - len(declining) - len(ascending), 'limbo')

In [ ]:
dfb = _dfb.join(df_quotes.loc[:, ['symbol', 'bidPrice', 'askPrice', 'volume']].set_index('symbol')).reset_index()
dfb['mid'] = (dfb.bidPrice + dfb.askPrice)/2
dfb['pctSpread'] = (dfb.askPrice - dfb.bidPrice)/dfb.mid*100
dfb['rank20'] = (dfb['mid'].astype(float) - dfb['MA20'].astype(float))/(dfb['UB20'] - dfb['LB20'])*200
dfb['rank30'] = (dfb['mid'].astype(float) - dfb['MA30'].astype(float))/(dfb['UB30'] - dfb['LB30'])*200
dfb['rank_d'] = dfb.rank20 - dfb.rank30

### Bollinger Ranking sorted by 20-day data

In [ ]:
__df = dfb.sort_values(by='rank20', ascending=False)
plot_metrics_in_one_row(__df.head(10), ['symbol'], ['rank20', 'rank30'], shared_y=True)
plot_metrics_in_one_row(__df.tail(10), ['symbol'], ['rank20', 'rank30'], shared_y=True, log_y_threshold=-500)

In [ ]:
__df = dfb[dfb.symbol.str.contains(r'^(?:SPY|QQQ|DIA|NVDA|META|MSFT|AAPL|TSLA|AMZN|GOOGL|COST|GLD|IBIT|AMD|ETHA|TSM|CRCL|TLT)')].sort_values(by='rank20')
plot_metrics_in_one_row(__df, ['symbol'], ['rank20', 'rank_d', 'rank30'], shared_y=False, log_y_threshold=-500)

In [ ]:
_symbol = 'CHTR'
_days=30
px.line(df_ema[_symbol].tail(_days).join(df_price.loc[:, [_symbol]].tail(_days)), title=_symbol, height=600)

In [ ]:
pd.DataFrame([(_, dfb.rank20.quantile(_), dfb.rank30.quantile(_)) for _ in [0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]], columns=['pctile', 'rank20', 'rank30'])

In [ ]:
px.histogram(dfb, x="rank20", nbins=20, title="Distribution of rank20",
    labels={'rank20': '20-day Bollinger'}, # Rename axis label
    template="plotly_white"
)

In [ ]:
px.histogram(dfb, x="rank30", nbins=20, title="Distribution of rank30",
    labels={'rank30': '30-day Bollinger'}, # Rename axis label
    template="plotly_white"
)